In [1]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from langchain_community.document_loaders import Docx2txtLoader

c:\coding\rag-agent\python\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Read all the docx files from the data directory
def process_all_docx_files(path):
    all_documents = []
    docx_dir = Path(path)
    # Find all the docx files in the directory
    docx_files = list(docx_dir.glob("**/*.docx"))
    print(f"Found {len(docx_files)} DOCX files to process")

    # Process each DOCX file
    for docx_file in docx_files:
        try:
            loader = Docx2txtLoader(str(docx_file))
            documents = loader.load()
            # add source information to metadata
            for doc in documents:
                doc.metadata["source"] = str(docx_file)
                doc.metadata["file_type"] = "docx"
            all_documents.extend(documents)
            print(f"Processed {docx_file.name}, found {len(documents)} documents")
        except Exception as e:
            print(f"Error processing {docx_file.name}: {e}")

    print(f"Successfully processed {len(all_documents)} documents")
    return all_documents

all_documents = process_all_docx_files("data")


Found 49 DOCX files to process
Processed algebra and geometry.docx, found 1 documents
Processed applied chemistry.docx, found 1 documents
Processed applied maths.docx, found 1 documents
Processed applied physics.docx, found 1 documents
Processed basic engineering drawing.docx, found 1 documents
Processed BEE.docx, found 1 documents
Processed c++.docx, found 1 documents
Processed c.docx, found 1 documents
Processed calculus I.docx, found 1 documents
Processed calculus II.docx, found 1 documents
Processed communication techniques.docx, found 1 documents
Processed computer architecture.docx, found 1 documents
Processed computer graphics.docx, found 1 documents
Processed data communication.docx, found 1 documents
Processed DBMS.docx, found 1 documents
Processed digital logic.docx, found 1 documents
Processed dsa.docx, found 1 documents
Processed electronic devices and circuits.docx, found 1 documents
Processed instrumentation.docx, found 1 documents
Processed MALP.docx, found 1 documents
P

In [3]:
print(all_documents)

[Document(metadata={'source': 'data\\PU computer engineering-20260127T133130Z-3-001\\algebra and geometry.docx', 'file_type': 'docx'}, page_content="Here's the extracted text from the images, broken down by question:\n\nPage 1\n\nPOKHARA UNIVERSITY Level: Bachelor Year: 2023 Programme: BE Semester: Spring Full Marks: 100 Course: Algebra and Geometry Pass Marks: 45 Time: 3 hrs. Candidates are required to give their answers in their own words as far as practicable. The figures in the margin indicate full marks. Attempt all the questions.\n\nCheck consistency and solve by Gauss elimination method: x + y + z = 1 2x + 3y + z = 2 x - 2y + 3z = 1\n\n\nSolve the linear programming problem by simplex method (constructing duality): Minimize Z = x1 + 9x1 + 9x2 subject to x1 + 4x1 + 2x2 ≤ 6, 3x1+ x2 ≤ 20, x1 ≥ 0, x2 ≥ 0. OR Using simplex method maximize Z = 150x1 + 100x2 subject to: x1 + 2x2 ≤ 8, x2 ≤ 4, x1 + x2 ≤ 6, x1 ≥ 0, x2 ≥ 0\n\n\na) Find the eigenvalues and eigenvectors of the matrix: A = [

In [4]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

embedding and vector store


In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb 
from chromadb.config import Settings
import uuid
from sklearn.metrics.pairwise import cosine_similarity


In [6]:
# handel embedding
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2", ):
        self.model_name = model_name
        self.model = None
        self.load_model()

    def load_model(self):
        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded successfully")
        except Exception as e:
            print(f"Error loading model: {e}")
    
    def generate_embedding(self,text):
        if self.model is None:
            raise ValueError("Model is not loaded")
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embedding shape", embeddings.shape)
        return embeddings

# initalize embedding manager
embedding_manager = EmbeddingManager()


Loading model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 763.24it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully


In [7]:
# vector store
class VectorStore:
    def __init__(self, collection_name="documents_collection", persist_directory="data/chroma_db"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self.initialize_store()

    def initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection of document embeddings"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for chromadb
        ids = []
        metadatas = []
        document_texts = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            document_texts.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=document_texts
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: documents_collection
Existing documents in collection: 0


In [8]:
# Process documents and add to vector store
# 1. Split documents into chunks
split_docs = split_documents(all_documents)

# 2. Generate embeddings for all chunks
print(f"\nGenerating embeddings for {len(split_docs)} chunks...")
texts = [doc.page_content for doc in split_docs]
embeddings = embedding_manager.generate_embedding(texts)

# 3. Add documents and embeddings to vector store
vectorstore.add_documents(split_docs, embeddings)

print(f"\nPipeline complete! Total documents in store: {vectorstore.collection.count()}")

Split 49 documents into 262 chunks

Example chunk:
Content: Here's the extracted text from the images, broken down by question:

Page 1

POKHARA UNIVERSITY Level: Bachelor Year: 2023 Programme: BE Semester: Spring Full Marks: 100 Course: Algebra and Geometry P...
Metadata: {'source': 'data\\PU computer engineering-20260127T133130Z-3-001\\algebra and geometry.docx', 'file_type': 'docx'}

Generating embeddings for 262 chunks...


Batches: 100%|██████████| 9/9 [00:08<00:00,  1.06it/s]


embedding shape (262, 384)
Adding 262 documents to vector store...
Successfully added 262 documents to vector store
Total documents in collection: 262

Pipeline complete! Total documents in store: 262


In [9]:
chunks = split_documents(all_documents)
chunks

Split 49 documents into 262 chunks

Example chunk:
Content: Here's the extracted text from the images, broken down by question:

Page 1

POKHARA UNIVERSITY Level: Bachelor Year: 2023 Programme: BE Semester: Spring Full Marks: 100 Course: Algebra and Geometry P...
Metadata: {'source': 'data\\PU computer engineering-20260127T133130Z-3-001\\algebra and geometry.docx', 'file_type': 'docx'}


[Document(metadata={'source': 'data\\PU computer engineering-20260127T133130Z-3-001\\algebra and geometry.docx', 'file_type': 'docx'}, page_content="Here's the extracted text from the images, broken down by question:\n\nPage 1\n\nPOKHARA UNIVERSITY Level: Bachelor Year: 2023 Programme: BE Semester: Spring Full Marks: 100 Course: Algebra and Geometry Pass Marks: 45 Time: 3 hrs. Candidates are required to give their answers in their own words as far as practicable. The figures in the margin indicate full marks. Attempt all the questions.\n\nCheck consistency and solve by Gauss elimination method: x + y + z = 1 2x + 3y + z = 2 x - 2y + 3z = 1\n\n\nSolve the linear programming problem by simplex method (constructing duality): Minimize Z = x1 + 9x1 + 9x2 subject to x1 + 4x1 + 2x2 ≤ 6, 3x1+ x2 ≤ 20, x1 ≥ 0, x2 ≥ 0. OR Using simplex method maximize Z = 150x1 + 100x2 subject to: x1 + 2x2 ≤ 8, x2 ≤ 4, x1 + x2 ≤ 6, x1 ≥ 0, x2 ≥ 0\n\n\na) Find the eigenvalues and eigenvectors of the matrix: A = [

In [10]:
# convert the text into embeddings 
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embedding(texts)
# store in vector database
vectorstore.add_documents(chunks, embeddings)

Batches: 100%|██████████| 9/9 [00:08<00:00,  1.09it/s]


embedding shape (262, 384)
Adding 262 documents to vector store...
Successfully added 262 documents to vector store
Total documents in collection: 524


In [15]:
# retrieve pipeline from vector store   

class RAGRetriever:
    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query, top_k=5, score_threshold=0.0, verbose=True):
        print(f"Retrieving top {top_k} documents for query: {query}")
        # generate query embedding
        query_embedding = self.embedding_manager.generate_embedding([query])[0]
        # search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                include=["documents", "metadatas", "distances"],
            )
            retrieved_docs = []
            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results.get("ids", [[None] * len(documents)])[0]
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance  # assuming distance is cosine distance
                    if verbose:
                        print(f"rank {i + 1}: sim={similarity_score:.3f} dist={distance:.3f} id={doc_id}")
                    if score_threshold is None or similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1,
                        })
                if verbose and not retrieved_docs:
                    print(f"All results were below score_threshold={score_threshold}")
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [16]:
rag_retriever.retrieve("questions from POKHARA UNIVERSITY Level: Bachelor Year: 2023 Programme: B.E. Semester: Spring Full Marks: 100 Course: Applied Physics")

Retrieving top 5 documents for query: questions from POKHARA UNIVERSITY Level: Bachelor Year: 2023 Programme: B.E. Semester: Spring Full Marks: 100 Course: Applied Physics


Batches: 100%|██████████| 1/1 [00:00<00:00, 39.84it/s]

embedding shape (1, 384)
rank 1: sim=0.475 dist=0.525 id=doc_cdd0c58e_8
rank 2: sim=0.475 dist=0.525 id=doc_963b2c1c_8
rank 3: sim=0.158 dist=0.842 id=doc_27dd38f7_236
rank 4: sim=0.158 dist=0.842 id=doc_32516eaa_236
rank 5: sim=0.144 dist=0.856 id=doc_1f94c16a_244
Retrieved 5 documents (after filtering)


[{'id': 'doc_cdd0c58e_8',
  'content': "Here's the text extracted from the images, divided into sections as presented in the document:\n\nPOKHARA UNIVERSITY\n\nLevel: Bachelor Semester: Spring Year: 2024 Programme: BE Course: Applied Chemistry Full Marks: 100 Pass Marks: 45 Time: 3 hrs.\n\nCandidates are required to give their own words as far as practicable. The figures in the margin indicate full marks. Attempt all the questions.",
  'metadata': {'content_length': 389,
   'file_type': 'docx',
   'doc_index': 8,
   'source': 'data\\PU computer engineering-20260127T133130Z-3-001\\applied chemistry.docx'},
  'similarity_score': 0.47521018981933594,
  'distance': 0.5247898101806641,
  'rank': 1},
 {'id': 'doc_963b2c1c_8',
  'content': "Here's the text extracted from the images, divided into sections as presented in the document:\n\nPOKHARA UNIVERSITY\n\nLevel: Bachelor Semester: Spring Year: 2024 Programme: BE Course: Applied Chemistry Full Marks: 100 Pass Marks: 45 Time: 3 hrs.\n\nCandi